# Data Understanding

## 0. Configuración

Se importan las librerías y se definen las rutas y constantes. La raíz del proyecto se busca subiendo desde el directorio de trabajo hasta encontrar la carpeta `Data/`, así que el notebook funciona tanto si se abre desde `src/DU/` como desde la raíz.

Nombres de los cuartos de la ubre que aparecen en todo el notebook:

| Código | Cuarto |
|---|---|
| `di` | delantero izquierdo |
| `dd` | delantero derecho |
| `ti` | trasero izquierdo |
| `td` | trasero derecho |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data").is_dir())
ENTRADA = RAIZ / "Data" / "dataset_vacas_anonimo.csv"
SALIDA = RAIZ / "Data" / "eda"
FIGURAS = SALIDA / "figuras"
DATASET_LIMPIO = RAIZ / "Data" / "dataset_vacas_limpio.csv"
RESUMEN = SALIDA / "resumen_eda.md"
FIGURAS.mkdir(parents=True, exist_ok=True)

# Cuartos de la ubre: delantero/trasero, izquierdo/derecho
CUARTOS = ["di", "dd", "ti", "td"]
NOMBRE_CUARTO = {"di": "Del. izq.", "dd": "Del. der.", "ti": "Tras. izq.", "td": "Tras. der."}

# Alertas que el robot reporta como texto con los cuartos afectados ("DI,TD", "Todos")
ALERTAS = {
    "estado_patada": "patada",
    "estado_incompleto": "incompleto",
    "estado_pezones_no_encontrados": "pezon_no_encontrado",
    "estado_amd_pezon": "amd_pezon",
}

MEDIDAS_CUARTO = {
    "producciones_kg": "Produccion (kg)",
    "media_de_los_flujos_kg_min": "Flujo medio (kg/min)",
    "conductividad_ms_cm": "Conductividad (mS/cm)",
    "flujos_maximos_kg_min": "Flujo maximo (kg/min)",
}

# Paleta de referencia (validada para daltonismo en el orden dado)
AZUL, NARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
TINTA, TINTA_SECUNDARIA, REJILLA = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.edgecolor": TINTA_SECUNDARIA, "axes.labelcolor": TINTA_SECUNDARIA,
    "xtick.color": TINTA_SECUNDARIA, "ytick.color": TINTA_SECUNDARIA, "text.color": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2,
})
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

## 1. Extracción

Cada fila del archivo es **una sesión del robot de ordeño**: una visita de una vaca al robot. El archivo se armó con `generate_data.py` (une los archivos de cada vaca) y `anonymize_data.py` (cambia los identificadores de vaca y de máquina por hashes).

Las columnas siguen el patrón `<grupo>_<medida>[_<cuarto>]`:

- `main_*`: datos generales de la sesión (hora de inicio, duración, producción, intervalo desde el ordeño anterior).
- `estado_*`: banderas y alertas del robot (número de ordeño del día, patada, ordeño incompleto, MDI, AMD).
- `producciones_kg_*`, `media_de_los_flujos_*`, `flujos_maximos_*`, `conductividad_*`, `sangre_ppm_*`: medidas **por cuarto** de la ubre.
- `misc_*`: robot que ordeñó (`misc_eo_po`), destino de la leche y razón de la desviación.

In [ ]:
original = pd.read_csv(ENTRADA, low_memory=False)
print(f"Filas: {len(original):,} | Columnas: {original.shape[1]}")
original.head()

In [ ]:
original.info()

## 2. Calidad del dataset original

Antes de transformar nada, conviene ver qué tan completo está el archivo y si los formatos son consistentes.

In [ ]:
def guardar(fig, nombre):
    """Guarda la figura en Data/eda/figuras/ y la muestra en el notebook."""
    fig.savefig(FIGURAS / nombre)
    plt.show()


nulos = (original.isna().mean() * 100).sort_values()
nulos = nulos[nulos > 0]
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(nulos.index, nulos.values, color=AZUL, height=0.7)
ax.set_xlabel("% de valores nulos")
ax.set_title("Valores nulos por columna (dataset original)")
ax.grid(axis="y", visible=False)
guardar(fig, "01_nulos_por_columna.png")

**Qué se observa en los nulos**

- `estado_rcs_1000_celulas_ml` (conteo de células somáticas) está **100 % vacía**, y `misc_usuario` tiene solo 2 valores. No aportan nada y se eliminan.
- Las alertas de texto (`estado_patada`, `estado_incompleto`, `estado_pezones_no_encontrados`, `estado_amd_pezon`) tienen entre 84 y 91 % de nulos. **Aquí un nulo significa "sin alerta"**, no un dato faltante, así que se convierten en banderas 0/1.
- `main_accion` está nula en el 97 % de las filas porque solo la traen los archivos CSV. En los XLSX todas las filas son ordeños.
- `estado_mdi` tiene alrededor de 10 % de nulos. Entre ellos están todas las filas de origen CSV, que no traen esa columna.
- Las medidas por cuarto tienen entre 2 y 4 % de nulos, y el cuarto trasero derecho (`td`) es el que más tiene.

In [ ]:
es_csv = original["main_hora_de_inicio"].str.contains(r"[ap]\. m\.", regex=True)
print("Filas con formato CSV:", es_csv.sum(), "| vacas:", original.loc[es_csv, "id_vaca"].nunique())
print("Filas con formato XLSX:", (~es_csv).sum(), "| vacas:", original.loc[~es_csv, "id_vaca"].nunique())
print("Vacas en ambas fuentes:",
      len(set(original.loc[es_csv, "id_vaca"]) & set(original.loc[~es_csv, "id_vaca"])))

display(pd.DataFrame({
    "CSV": original.loc[es_csv, ["main_hora_de_inicio", "main_duracion_mm_ss"]].iloc[0],
    "XLSX": original.loc[~es_csv, ["main_hora_de_inicio", "main_duracion_mm_ss"]].iloc[0],
}))
intervalo = original["main_intervalo_de_ordeno_hh_mm"].dropna()
print("Intervalos guardados como fecha de Excel:", intervalo.str.contains("-").sum())
print(intervalo[intervalo.str.contains("-")].head(3).tolist())

**Problemas de formato encontrados**

1. **Dos formatos de fecha y duración.** Los registros de los CSV usan `01/03/2025 05:37 a. m.` y duración `mm:ss`. Los de los XLSX usan `2025-03-01 05:37:12` y duración `hh:mm:ss`.
2. **Sesiones repetidas.** 28 vacas aparecen en las dos fuentes, y entre marzo y julio de 2025 la misma sesión viene dos veces. El CSV solo trae la hora al minuto, así que la sesión repetida no se ve como duplicado exacto.
3. **Intervalos de 24 h o más guardados como fecha de Excel.** Excel convirtió esos intervalos en una fecha: `1900-01-01 00:59:02` equivale a 24 h 59 min.

## 3. Transformación (ETL)

Las funciones de esta sección corrigen los problemas anteriores y agregan las variables derivadas que se usan en el análisis.

| Paso | Qué se hace | Por qué |
|---|---|---|
| Fechas | Se unifican los dos formatos en `fecha_hora` | Para poder ordenar y agrupar por tiempo |
| Duración | Se convierte a minutos (`duracion_min`) | `mm:ss` y `hh:mm:ss` no se pueden comparar como texto |
| Intervalo | Se convierte a horas (`intervalo_h`), sumando 24 h por cada día de la fecha de Excel | Para recuperar los intervalos de más de 24 h |
| Columnas vacías | Se eliminan `estado_rcs_1000_celulas_ml` y `misc_usuario` | No tienen información |
| Duplicados | Se quitan los duplicados exactos y las sesiones repetidas (misma vaca, mismo minuto); se conserva la del XLSX | El XLSX trae segundos, MDI e intervalo; el CSV no |
| Conductividad 0 | Se pasa a nulo | La leche no puede tener 0 mS/cm: es una lectura fallida del sensor |
| Alertas | Se crean banderas `patada_di`, `incompleto_td`, …, `*_alguno` | Para poder contarlas y correlacionarlas |
| Derivadas | `hora`, `mes`, `dias_en_registro`, producción delantera y trasera, conductividad máxima, sangre máxima | Para resumir la ubre en una sola variable |

In [ ]:
def parsear_fecha(serie):
    """
        Convierte la hora de inicio a datetime. Los registros que venian de CSV usan
        "01/03/2025 05:37 a. m." y los de XLSX usan "2025-03-01 05:37:12".
    """
    es_csv = serie.str.contains(r"[ap]\. m\.", regex=True)
    texto = serie.str.replace("a. m.", "AM", regex=False).str.replace("p. m.", "PM", regex=False)
    fechas_csv = pd.to_datetime(texto.where(es_csv), format="%d/%m/%Y %I:%M %p", errors="coerce")
    fechas_xlsx = pd.to_datetime(texto.where(~es_csv), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    return fechas_csv.fillna(fechas_xlsx)


def duracion_a_minutos(serie):
    """Convierte la duracion del ordeno a minutos. Acepta "mm:ss" (CSV) y "hh:mm:ss" (XLSX)."""
    partes = serie.str.split(":", expand=True).astype(float)
    if partes.shape[1] == 2:
        return partes[0] + partes[1] / 60
    con_horas = partes[2].notna()
    horas = partes[0].where(con_horas, 0)
    minutos = partes[1].where(con_horas, partes[0])
    segundos = partes[2].where(con_horas, partes[1])
    return horas * 60 + minutos + segundos / 60


def intervalo_a_horas(serie):
    """
        Convierte el intervalo entre ordenos a horas. Excel guarda los intervalos de 24 h o
        mas como fecha ("1900-01-01 00:59:02" equivale a 24 h 59 min), asi que los dias
        desde el 31/12/1899 se suman como horas.
    """
    con_fecha = serie.str.contains("-", na=False)
    tiempo = serie.str.split(" ").str[-1]
    partes = tiempo.str.split(":", expand=True).astype(float)
    horas = partes[0] + partes[1] / 60 + partes[2] / 3600

    dias = (pd.to_datetime(serie.where(con_fecha).str[:10], errors="coerce")
            - pd.Timestamp("1899-12-31")).dt.days
    return horas + dias.fillna(0) * 24


def separar_alertas(df):
    """
        Crea una bandera 0/1 por cuarto para cada alerta de texto, por ejemplo
        estado_patada = "DI,TD" -> patada_di = 1, patada_td = 1.
    """
    for columna, nombre in ALERTAS.items():
        texto = df[columna].fillna("").str.upper()
        todos = texto.eq("TODOS")
        for cuarto in CUARTOS:
            df[f"{nombre}_{cuarto}"] = (todos | texto.str.contains(cuarto.upper())).astype(int)
        df[f"{nombre}_alguno"] = texto.ne("").astype(int)
    return df

In [ ]:
def transformar(df):
    """
        Limpia el dataset y agrega variables derivadas para el analisis.

        Returns:
            tuple: (DataFrame limpio, dict con lo que se hizo en cada paso)
    """
    bitacora = {"filas_originales": len(df)}

    vacias = [c for c in df.columns
              if df[c].isna().all() or (df[c].nunique() <= 1 and df[c].isna().mean() > 0.99)]
    df = df.drop(columns=vacias)
    bitacora["columnas_eliminadas"] = vacias

    df["origen"] = np.where(df["main_hora_de_inicio"].str.contains(r"[ap]\. m\.", regex=True), "csv", "xlsx")
    df["fecha_hora"] = parsear_fecha(df["main_hora_de_inicio"])
    df["duracion_min"] = duracion_a_minutos(df["main_duracion_mm_ss"])
    df["intervalo_h"] = intervalo_a_horas(df["main_intervalo_de_ordeno_hh_mm"])
    # Los XLSX no traen la accion: todos sus registros son ordenos
    df["main_accion"] = df["main_accion"].fillna("Ordeño")
    df = df.drop(columns=["main_hora_de_inicio", "main_duracion_mm_ss", "main_intervalo_de_ordeno_hh_mm"])

    bitacora["duplicados_exactos"] = int(df.duplicated().sum())
    df = df.drop_duplicates()

    # Las vacas que estan en ambas fuentes repiten sesiones: el CSV solo trae minutos,
    # asi que se compara vaca + minuto de inicio y se conserva el registro del XLSX
    df["_minuto"] = df["fecha_hora"].dt.floor("min")
    df = df.sort_values(["id_vaca", "_minuto", "origen"], ascending=[True, True, False])
    repetidas = df.duplicated(["id_vaca", "_minuto"], keep="first")
    bitacora["sesiones_repetidas_entre_fuentes"] = int(repetidas.sum())
    df = df.loc[~repetidas].drop(columns="_minuto").sort_values(["id_vaca", "fecha_hora"])

    # Una conductividad de 0 mS/cm no es fisicamente posible en leche: es lectura faltante
    for cuarto in CUARTOS:
        columna = f"conductividad_ms_cm_{cuarto}"
        bitacora[f"conductividad_cero_{cuarto}"] = int((df[columna] == 0).sum())
        df[columna] = df[columna].replace(0, np.nan)

    df = separar_alertas(df)
    df["fecha"] = df["fecha_hora"].dt.normalize()
    df["hora"] = df["fecha_hora"].dt.hour
    df["mes"] = df["fecha_hora"].dt.month
    df["anio"] = df["fecha_hora"].dt.year
    df["dia_semana"] = df["fecha_hora"].dt.dayofweek
    df["dias_en_registro"] = (df["fecha"] - df.groupby("id_vaca")["fecha"].transform("min")).dt.days
    df["produccion_delantera_kg"] = df["producciones_kg_di"] + df["producciones_kg_dd"]
    df["produccion_trasera_kg"] = df["producciones_kg_ti"] + df["producciones_kg_td"]
    df["conductividad_max_ms_cm"] = df[[f"conductividad_ms_cm_{c}" for c in CUARTOS]].max(axis=1)
    df["sangre_max_ppm"] = df[[f"sangre_ppm_{c}" for c in CUARTOS]].max(axis=1)

    bitacora["filas_finales"] = len(df)
    return df.reset_index(drop=True), bitacora


df, bitacora = transformar(original.copy())
pd.Series(bitacora, name="valor").to_frame()

**Resultado de la limpieza**

- De 286,217 filas quedan **280,906**. Se quitaron 2 duplicados exactos y **5,309 sesiones repetidas** entre CSV y XLSX. De los CSV solo se conservan las ~1,900 sesiones que no estaban en los XLSX.
- **La conductividad del cuarto trasero derecho tiene 79,300 lecturas en 0**, contra 4,636 del delantero izquierdo. Es un problema del sensor de ese cuarto. Al pasarlas a nulo, `conductividad_ms_cm_td` queda con alrededor de 32 % de faltantes.

## 4. Visualizar y consultar los datos

Una vez limpio, se revisa la forma general del dataset: periodo cubierto, tipos de dato, nulos que quedan, estadísticos y variables categóricas.

In [ ]:
print(f"Registros: {len(df):,} | Columnas: {df.shape[1]} | Vacas: {df['id_vaca'].nunique()}")
print(f"Periodo: {df['fecha_hora'].min():%d/%m/%Y} a {df['fecha_hora'].max():%d/%m/%Y}")
df.head()

In [ ]:
tipos = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(),
})
tipos[tipos["nulos_%"] > 0].sort_values("nulos_%", ascending=False)

In [ ]:
principales = ["main_produccion_kg", "duracion_min", "intervalo_h", "estado_mdi",
               "conductividad_max_ms_cm", "sangre_max_ppm", "estado_numero_de_ordeno"]
df[principales].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

**Lectura de los estadísticos**

- **Producción por ordeño**: mediana de 14.3 kg y 98 % de las sesiones entre 3.9 y 29.3 kg. El máximo de 78 kg es muy atípico y probablemente es un error de registro.
- **Duración**: mediana de 7.6 min. El máximo de 55.8 min también es atípico.
- **Intervalo entre ordeños**: mediana de 9 h, lo que da unos 2.3 ordeños por vaca al día. Como el robot es de **ordeño voluntario**, cada vaca va a su propio ritmo. El máximo de 2,932 h (unos 4 meses) no es un intervalo real: es el primer ordeño después de un periodo seco y debe tratarse como valor atípico.
- **MDI** (índice de mastitis): casi todas las sesiones están cerca de 1.1 y hay una cola larga hasta 15.
- **Sangre**: el percentil 75 es 0 ppm, así que la sangre es un **evento raro**, pero cuando aparece llega a miles de ppm.

In [ ]:
for columna in ["main_accion", "misc_destino_leche", "misc_razon_de_la_desviacion", "misc_eo_po", "origen"]:
    display(df[columna].value_counts(dropna=False).to_frame())

**Variables categóricas**

- `main_accion`: casi todo son ordeños. Solo hay 168 sesiones "Rechazada" y 41 "Echada sin ordeñar", y ninguna de ellas tiene producción.
- `misc_destino_leche`: 94 % de la leche va a **Tanque**. El resto se desvía a **Drenaje** (unas 10,400 sesiones) o **Divert 3** (unas 5,400). "Divert 1" y "Divert 2" casi no aparecen.
- `misc_razon_de_la_desviacion` solo se registra en unas 1,500 sesiones, casi siempre "Sangre por pezón por encima del umbral". La mayoría de las desviaciones a drenaje **no traen razón**.
- `misc_eo_po`: hay **3 robots** (anonimizados) con carga muy parecida, de 91 a 98 mil sesiones cada uno. El cuarto valor solo tiene 2 sesiones.

In [ ]:
por_vaca = df.groupby("id_vaca").agg(
    registros=("fecha_hora", "size"),
    primer_registro=("fecha", "min"),
    ultimo_registro=("fecha", "max"),
    dias_con_ordeno=("fecha", "nunique"),
    produccion_media_kg=("main_produccion_kg", "mean"),
)
por_vaca["ordenos_por_dia"] = por_vaca["registros"] / por_vaca["dias_con_ordeno"]
print(f"Vacas con menos de 30 dias de registros: {(por_vaca['dias_con_ordeno'] < 30).sum()}")
por_vaca.describe().T.round(2)

In [ ]:
def histograma(ax, serie, titulo_eje, color=AZUL, recorte=(0.005, 0.995)):
    """Histograma recortado a los percentiles dados para que los extremos no aplasten la forma."""
    serie = serie.dropna()
    bajo, alto = serie.quantile(recorte[0]), serie.quantile(recorte[1])
    ax.hist(serie.clip(bajo, alto), bins=60, color=color, edgecolor="white", linewidth=0.5)
    ax.axvline(serie.median(), color=TINTA, linestyle="--", linewidth=1.2)
    ax.text(0.98, 0.95, f"mediana {serie.median():.2f}", transform=ax.transAxes, ha="right", va="top")
    ax.set_title(titulo_eje)
    ax.set_ylabel("Sesiones")


fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
mensual = df.groupby(df["fecha"].dt.to_period("M"))["id_vaca"].nunique()
ejes[0].bar(mensual.index.to_timestamp(), mensual.values, width=25, color=AZUL)
ejes[0].set_title("Vacas con registros por mes")
ejes[0].set_ylabel("Vacas")
ejes[0].tick_params(axis="x", rotation=45)

histograma(ejes[1], por_vaca["dias_con_ordeno"], "Dias con ordeno por vaca", recorte=(0, 1))
ejes[1].set_ylabel("Vacas")
fig.tight_layout()
guardar(fig, "02_cobertura_temporal.png")

**Cobertura**

- **El hato crece de forma constante**: unas 34 vacas en julio de 2022 y más de 180 en enero de 2026. Hay que tenerlo en cuenta en cualquier serie de tiempo **agregada**. Por eso, desde aquí la producción se mide por vaca y no como total del hato.
- **El historial por vaca es muy desigual**: de 2 a 1,164 días, con mediana de 520. Hay 7 vacas con menos de 30 días, demasiado poco para construir una línea base individual.

## 5. Visualizar patrones y distribuciones

### 5.1 Distribuciones de las variables principales

Los histogramas se recortan al rango 0.5–99.5 % para que los valores extremos no aplasten la forma. La línea punteada marca la mediana.

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(15, 8))
variables = [
    ("main_produccion_kg", "Produccion por ordeno (kg)"),
    ("duracion_min", "Duracion del ordeno (min)"),
    ("intervalo_h", "Intervalo entre ordenos (h)"),
    ("estado_mdi", "MDI (indicador de mastitis)"),
    ("conductividad_max_ms_cm", "Conductividad maxima entre cuartos (mS/cm)"),
]
for ax, (columna, nombre) in zip(ejes.flat, variables):
    histograma(ax, df[columna], nombre)

conteo = df["estado_numero_de_ordeno"].value_counts().sort_index()
ejes[1, 2].bar(conteo.index.astype(int).astype(str), conteo.values, color=AZUL, width=0.6)
ejes[1, 2].set_title("Numero de ordeno del dia")
ejes[1, 2].set_ylabel("Sesiones")
fig.suptitle("Distribucion de las variables principales (recortadas al 0.5-99.5 %)", y=1.01)
fig.tight_layout()
guardar(fig, "03_distribuciones_principales.png")

**Qué se observa**

- **Producción**: forma de campana con un ligero sesgo a la derecha. No hay grupos separados.
- **Duración e intervalo**: sesgados a la derecha, con colas largas. Para modelar conviene aplicar logaritmo o usar métodos robustos.
- **MDI**: **dos grupos**. Uno grande cerca de 1.0–1.2 y otro pequeño entre 1.5 y 2.0. Ese segundo grupo puede ser el de las sesiones con inflamación y es candidato a etiqueta de anomalía.
- **Conductividad máxima**: casi normal, centrada en 5.1 mS/cm, con cola arriba de 6.5. La conductividad sube cuando hay mastitis.
- **Número de ordeño del día**: casi siempre 1 a 3. Las vacas con 4 o más ordeños en un día son pocas, coherente con el ordeño voluntario.

### 5.2 Comparación entre cuartos de la ubre

El robot mide cada cuarto por separado. Si los cuartos no son iguales entre sí, una anomalía debe buscarse comparando cada cuarto contra su propio historial y no contra el promedio de la ubre.

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (prefijo, nombre) in zip(ejes.flat, MEDIDAS_CUARTO.items()):
    datos = [df[f"{prefijo}_{c}"].dropna() for c in CUARTOS]
    caja = ax.boxplot(datos, tick_labels=[NOMBRE_CUARTO[c] for c in CUARTOS], showfliers=False,
                      patch_artist=True, medianprops={"color": TINTA})
    for parche, color in zip(caja["boxes"], [AZUL, AZUL, NARANJA, NARANJA]):
        parche.set_facecolor(color)
        parche.set_alpha(0.75)
    ax.set_title(nombre)
    ax.grid(axis="x", visible=False)
fig.suptitle("Medidas por cuarto de la ubre (azul: delanteros, naranja: traseros)", y=1.01)
fig.tight_layout()
guardar(fig, "04_medidas_por_cuarto.png")

**Qué se observa**

- **Los traseros producen más**: mediana de unos 4.0 kg contra unos 3.4 kg de los delanteros, y más variabilidad.
- **El flujo medio y el máximo son parecidos** en los cuatro cuartos. Los traseros producen más porque se ordeñan más tiempo, no porque su flujo sea mayor.
- **La conductividad es casi igual entre cuartos** (unos 4.9 mS/cm). Por eso, si un cuarto se aleja de los otros en la misma sesión, es buena señal de un problema localizado en ese cuarto.

### 5.3 Producción a lo largo del tiempo

Como el hato crece, la producción se normaliza por vacas ordeñadas (kg/vaca/día). Se descarta el último día porque los datos terminan a media mañana.

In [ ]:
diario = df.groupby("fecha").agg(produccion=("main_produccion_kg", "sum"), vacas=("id_vaca", "nunique"))
diario["kg_por_vaca"] = diario["produccion"] / diario["vacas"]
diario = diario[diario["vacas"] >= 5].iloc[:-1]

fig, ejes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ejes[0].plot(diario.index, diario["kg_por_vaca"], color=AZUL, alpha=0.25, linewidth=1)
ejes[0].plot(diario.index, diario["kg_por_vaca"].rolling(30, center=True).mean(), color=AZUL)
ejes[0].set_ylabel("kg / vaca / dia")
ejes[0].set_title("Produccion diaria por vaca (linea: media movil de 30 dias)")
ejes[1].plot(diario.index, diario["vacas"], color=TINTA_SECUNDARIA, linewidth=1.5)
ejes[1].set_ylabel("Vacas ordenadas")
ejes[1].set_title("Vacas ordenadas por dia")
fig.tight_layout()
guardar(fig, "05_serie_temporal_produccion.png")

**Qué se observa**

- La producción por vaca se mantiene entre **32 y 39 kg/vaca/día**, con ondas de varios meses. No crece al mismo ritmo que el hato, así que el aumento de producción total se explica por tener más vacas.
- Hay bajas claras **a finales del verano y en el otoño** (agosto–octubre de 2023 y de 2024) y picos **en invierno**. Esto se revisa por mes en la siguiente sección.
- Algunos días aislados caen mucho. Pueden ser días con datos incompletos o con fallas del robot y vale la pena revisarlos en la preparación de datos.

### 5.4 Patrones por hora, por mes y por intervalo

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.5))

por_hora = df.groupby("hora").size()
ejes[0].bar(por_hora.index, por_hora.values, color=AZUL, width=0.8)
ejes[0].set_title("Sesiones por hora del dia")
ejes[0].set_xlabel("Hora")
ejes[0].set_xticks(range(0, 24, 3))

meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
diario_vaca = df.groupby(["id_vaca", "fecha"])["main_produccion_kg"].sum().reset_index()
diario_vaca = diario_vaca[diario_vaca["main_produccion_kg"] > 0]
por_mes = diario_vaca.groupby(diario_vaca["fecha"].dt.month)["main_produccion_kg"].median()
ejes[1].plot(por_mes.index, por_mes.values, color=AZUL, marker="o", markersize=7)
ejes[1].set_xticks(range(1, 13), meses)
ejes[1].set_title("Mediana de kg/vaca/dia por mes del año")
ejes[1].set_ylabel("kg")

con_intervalo = df[df["intervalo_h"].between(4, 24)]
por_intervalo = con_intervalo.groupby(con_intervalo["intervalo_h"].astype(int))["main_produccion_kg"].median()
ejes[2].plot(por_intervalo.index, por_intervalo.values, color=NARANJA, marker="o", markersize=7)
ejes[2].set_title("Produccion por ordeno vs horas desde el anterior")
ejes[2].set_xlabel("Intervalo (h)")
ejes[2].set_ylabel("Mediana kg")
fig.tight_layout()
guardar(fig, "06_patrones_hora_mes_intervalo.png")

**Qué se observa**

- **Hora del día**: el robot trabaja las 24 horas, pero hay **tres caídas claras, cerca de las 4, 12 y 20 h**. Parecen pausas fijas, por ejemplo la limpieza del robot o el manejo del establo (alimentación, arreo). No las explica `misc_programa_de_lavado`, que aparece igual en todas las horas, así que hay que confirmarlo con el cliente. Fuera de esas horas las vacas visitan el robot de forma pareja.
- **Mes del año**: la producción por vaca es más alta en **diciembre–marzo** y más baja en **agosto–septiembre**, con unos 10 % de diferencia. Puede deberse a estrés por calor y humedad, pero también a que en cada mes hay vacas en distinta etapa de lactancia. Para distinguirlo hay que cruzar con el clima de NASA POWER.
- **Intervalo**: la producción **crece casi en línea recta** entre 4 y 15 h (de unos 10 a unos 19 kg) y se estabiliza después de unas 16 h. Es el patrón más fuerte del dataset: **un ordeño "bajo" puede ser normal si el intervalo fue corto**.

### 5.5 Alertas operativas por cuarto

Porcentaje de sesiones en que el robot marcó cada alerta en cada cuarto.

In [ ]:
tasa_alertas = pd.DataFrame(
    [[df[f"{n}_{c}"].mean() * 100 for c in CUARTOS] for n in ALERTAS.values()],
    index=list(ALERTAS.values()), columns=[NOMBRE_CUARTO[c] for c in CUARTOS],
)
fig, ax = plt.subplots(figsize=(8, 4))
imagen = ax.imshow(tasa_alertas.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(tasa_alertas.columns)), tasa_alertas.columns)
ax.set_yticks(range(len(tasa_alertas.index)), tasa_alertas.index)
ax.grid(False)
for i in range(tasa_alertas.shape[0]):
    for j in range(tasa_alertas.shape[1]):
        valor = tasa_alertas.iat[i, j]
        ax.text(j, i, f"{valor:.1f}%", ha="center", va="center",
                color="white" if valor > tasa_alertas.values.max() * 0.6 else TINTA)
fig.colorbar(imagen, ax=ax, label="% de sesiones")
ax.set_title("Frecuencia de alertas por cuarto")
guardar(fig, "07_alertas_por_cuarto.png")

**Qué se observa**

- **Ordeño incompleto** y **pezón no encontrado** pasan en los cuartos **traseros** más del doble de veces que en los delanteros (alrededor de 5.5 % contra 2 %). Coincide con que el brazo del robot tiene más dificultad para colocar las pezoneras traseras.
- **Las patadas siguen un patrón cruzado**: son más frecuentes en el **trasero derecho (7.2 %)** y el **delantero izquierdo (5.8 %)**. Puede deberse al lado por el que entra el brazo del robot.
- **AMD de pezón** es parecida en los cuatro cuartos (2.4–3.1 %), lo que sugiere que no depende de la posición del cuarto.

### 5.6 Correlaciones

Se usa la correlación de **Spearman** porque varias variables están sesgadas o son binarias y Spearman no supone relaciones lineales. Solo se rotulan los valores con |r| ≥ 0.2.

In [ ]:
columnas = ["main_produccion_kg", "duracion_min", "intervalo_h", "estado_numero_de_ordeno",
            "estado_mdi", "estado_amd_ubre", "conductividad_max_ms_cm", "sangre_max_ppm",
            "media_de_los_flujos_kg_min_di", "flujos_maximos_kg_min_di",
            "patada_alguno", "incompleto_alguno", "pezon_no_encontrado_alguno", "dias_en_registro"]
correlacion = df[columnas].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8.5))
imagen = ax.imshow(correlacion.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(columnas)), columnas, rotation=60, ha="right")
ax.set_yticks(range(len(columnas)), columnas)
ax.grid(False)
for i in range(len(columnas)):
    for j in range(len(columnas)):
        valor = correlacion.iat[i, j]
        if i != j and abs(valor) >= 0.2:
            ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=7.5,
                    color="white" if abs(valor) > 0.6 else TINTA)
fig.colorbar(imagen, ax=ax, label="Spearman")
ax.set_title("Correlacion entre variables (se rotulan |r| >= 0.2)")
guardar(fig, "08_correlaciones.png")

**Qué se observa**

- **Producción ↔ intervalo (0.54)**: es la relación más fuerte con la producción y confirma el patrón de la sección 5.4.
- **MDI ↔ AMD ubre (0.59) y MDI ↔ conductividad (0.56)**: los tres indicadores de salud de la ubre se mueven juntos. Son el núcleo para detectar anomalías.
- **MDI ↔ intervalo (0.43)**: las vacas que tardan más en volver al robot tienen MDI más alto. Puede ser un efecto de cómo el robot calcula el MDI o una señal de que las vacas enfermas visitan menos el robot.
- **Ordeño incompleto ↔ pezón no encontrado (0.77)**: casi siempre ocurren juntos, porque si el robot no encuentra el pezón no puede terminar el ordeño. Ambos **bajan la producción** (−0.32 y −0.30).
- **Flujo medio ↔ flujo máximo (0.94)**: miden casi lo mismo. Para modelar basta con una de las dos.
- **La sangre casi no se correlaciona con nada**, porque es un evento raro: en el 75 % de las sesiones vale 0.

### 5.7 Indicadores de salud de la ubre (posible mastitis)

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.5))
muestra = df.dropna(subset=["estado_mdi", "conductividad_max_ms_cm"])
muestra = muestra[muestra["estado_mdi"] < muestra["estado_mdi"].quantile(0.995)]
hexagonos = ejes[0].hexbin(muestra["conductividad_max_ms_cm"], muestra["estado_mdi"],
                           gridsize=50, cmap="Blues", mincnt=1, bins="log")
fig.colorbar(hexagonos, ax=ejes[0], label="Sesiones (log)")
ejes[0].set_xlabel("Conductividad maxima (mS/cm)")
ejes[0].set_ylabel("MDI")
ejes[0].set_title("MDI vs conductividad")
ejes[0].grid(False)

grupos = [df.loc[df["estado_amd_ubre"] == valor, "conductividad_max_ms_cm"].dropna() for valor in (0, 1)]
ejes[1].boxplot(grupos, tick_labels=["Sin AMD ubre", "Con AMD ubre"], showfliers=False,
                medianprops={"color": TINTA})
ejes[1].set_title("Conductividad segun alerta AMD de ubre")
ejes[1].set_ylabel("mS/cm")

destino = df["misc_destino_leche"].value_counts()
ejes[2].barh(destino.index[::-1], destino.values[::-1], color=AZUL, height=0.6)
ejes[2].set_xscale("log")
ejes[2].set_title("Destino de la leche (escala log)")
ejes[2].set_xlabel("Sesiones")
ejes[2].grid(axis="y", visible=False)
fig.tight_layout()
guardar(fig, "09_indicadores_mastitis.png")

**Qué se observa**

- **MDI vs conductividad**: la mayoría de las sesiones están juntas en MDI de unos 1.05 y conductividad de unos 5 mS/cm, la zona "normal". Hay una **segunda nube** con MDI de 1.5 a 2 y conductividad de 5 a 7. Ese es el grupo que conviene estudiar como posible anomalía.
- **Las sesiones con alerta AMD de ubre** tienen la conductividad claramente más alta: mediana de unos 5.8 mS/cm contra unos 5.0, y las cajas casi no se traslapan.
- **Destino de la leche**: la escala logarítmica muestra que Drenaje y Divert 3 son las únicas desviaciones frecuentes.

### 5.8 Curvas de producción por vaca

Para ver el comportamiento individual se grafica la producción diaria de las 3 vacas con más registros. Los días sin ordeño se dejan vacíos para que la línea se corte y no una con una recta los periodos sin datos.

In [ ]:
top = df["id_vaca"].value_counts().head(3).index
fig, ax = plt.subplots(figsize=(13, 4.5))
for vaca, color in zip(top, [AZUL, NARANJA, AQUA]):
    diario_una = df[df["id_vaca"] == vaca].groupby("fecha")["main_produccion_kg"].sum().asfreq("D")
    ax.plot(diario_una.index, diario_una.rolling(7, center=True, min_periods=4).mean(), color=color, label=vaca)
ax.legend(title="Vaca (media movil 7 dias)", frameon=False)
ax.set_ylabel("kg / dia")
ax.set_title("Produccion diaria de las 3 vacas con mas registros")
guardar(fig, "10_curvas_por_vaca.png")

**Qué se observa**

- Cada vaca muestra **ciclos de lactancia** claros. Después del parto la producción sube rápido hasta un pico de 55 a 70 kg/día y luego baja poco a poco durante meses.
- Los **huecos de varias semanas** entre ciclos son los **periodos secos** antes del siguiente parto. En todo el hato hay 267 huecos de 21 días o más, con mediana de 80 días. El primer ordeño después de un hueco es el que da los intervalos atípicos de miles de horas que se vieron en la sección 4.
- El dataset **no trae la fecha de parto**, pero se puede **estimar con estos huecos**. Con eso se calcularían los *días en leche*, que probablemente explican buena parte de la variación en producción.

## 6. Guardar resultados

Se guardan el dataset limpio y el resumen en Markdown. Al final se imprime la fila para registrar el evento en `docs/data_audit_logs.csv`. Hay que cambiar `<usuario>` por tu usuario de GitHub o Discord y pegar la fila debajo del encabezado.

In [ ]:
lineas = [
    "# Resumen de la exploracion inicial",
    "",
    f"- Registros limpios: {len(df):,} de {bitacora['filas_originales']:,} originales",
    f"- Vacas: {df['id_vaca'].nunique()}",
    f"- Periodo: {df['fecha_hora'].min():%d/%m/%Y} a {df['fecha_hora'].max():%d/%m/%Y}",
    f"- Duplicados exactos eliminados: {bitacora['duplicados_exactos']}",
    f"- Sesiones repetidas entre CSV y XLSX eliminadas: {bitacora['sesiones_repetidas_entre_fuentes']}",
    f"- Columnas vacias eliminadas: {', '.join(bitacora['columnas_eliminadas'])}",
    "",
    "## Figuras",
    "",
]
lineas += [f"- figuras/{f.name}" for f in sorted(FIGURAS.glob("*.png"))]
RESUMEN.write_text("\n".join(lineas) + "\n", encoding="utf-8")

df.to_csv(DATASET_LIMPIO, index=False)
print(f"Guardado en {DATASET_LIMPIO.relative_to(RAIZ)} y resumen en {RESUMEN.relative_to(RAIZ)}")

print("\nFila para docs/data_audit_logs.csv:")
print(f"{pd.Timestamp.now():%d/%m/%Y %H:%M}, <usuario>, vacas/3.1, Copia, "
      f"Exploracion inicial y limpieza basica del dataset anonimizado, Completado, "
      f"{bitacora['filas_originales'] - bitacora['filas_finales']} filas duplicadas retiradas; "
      f"conductividades en 0 pasadas a nulo; salida en {DATASET_LIMPIO.relative_to(RAIZ)}")

## 7. Conclusiones

1. **Calidad**: los datos sirven para el objetivo, pero necesitaron limpieza. Se unificaron dos formatos, se quitaron 5,311 sesiones duplicadas, se eliminaron 2 columnas vacías y se encontró un sensor de conductividad (trasero derecho) con alrededor de 29 % de lecturas en 0.
2. **Patrones principales**: la producción depende sobre todo del **intervalo entre ordeños** y de la **etapa de lactancia**, y en menor medida de la época del año. Los cuartos traseros producen más y fallan más en la colocación.
3. **Señales de anomalía**: **MDI, conductividad y AMD de ubre** se mueven juntos y forman un grupo de sesiones claramente separado del grupo normal. El **destino de la leche** sirve como etiqueta débil para validar.
4. **Siguientes pasos** (Data Preparation): crear `kg_por_hora` y los días en leche, decidir cómo tratar la conductividad del cuarto TD, integrar el clima y definir las líneas base por vaca y por cuarto.